# 01 — The Semantic Layer and Metric Views

**Time: about 15 minutes.**
Covers: Inconsistent metrics · Grain and fan-out · Unity Catalog metric views · `MEASURE()` · Synonyms and comments

### What you will be able to do afterwards

- Explain why two competent people get two different "revenue" numbers from the same tables.
- Define metrics once in a metric view (`CREATE VIEW ... WITH METRICS LANGUAGE YAML`) with joins, measures, dimensions, a global filter, synonyms and comments.
- Query it with `MEASURE()` and show that any breakdown re-aggregates to the same governed number.

In [ ]:
from helpers import utils, semantic_model

cfg = utils.use_lab_schema()
spark = utils.spark
print(f"Working in {cfg['catalog']}.{cfg['schema']}")

## Why raw tables produce inconsistent metrics

A metric is a **definition**, not a column: which rows count, at which grain, after which
exclusions, over which window. Raw tables hold the ingredients but not the recipe, so every
dashboard, notebook and AI assistant re-derives the recipe on its own, a little differently each time.

A **semantic layer** stores the recipe once, next to the data, and makes every consumer use it.
In Databricks that is a **Unity Catalog metric view**: a governed object whose YAML definition
separates *measures* (how to aggregate) from *dimensions* (what you may group and filter by).
The engine generates the correct SQL for whatever grouping the consumer asks for.

## Step 1 — Meet the data

In [ ]:
%sql
SELECT * FROM lab_orders LIMIT 5

In [ ]:
%sql
SELECT * FROM lab_refunds ORDER BY order_id LIMIT 8

## Step 2 — "What was our revenue in the last 90 days?"

Two analysts answer the CFO's question. Both queries look reasonable. Run them.

In [ ]:
%sql
-- Sales Ops dashboard: revenue = what customers paid, minus cancellations
SELECT ROUND(SUM(order_total), 2) AS revenue_90d
FROM lab_orders
WHERE status <> 'cancelled'
  AND order_date >= current_date() - INTERVAL 90 DAYS

In [ ]:
%sql
-- Finance notebook: revenue = order totals net of refunds
SELECT ROUND(SUM(o.order_total) - SUM(COALESCE(r.refund_amount, 0)), 2) AS revenue_90d
FROM lab_orders o
LEFT JOIN lab_refunds r ON o.order_id = r.order_id
WHERE o.status <> 'cancelled'
  AND o.order_date >= current_date() - INTERVAL 90 DAYS

Two numbers. They differ by tens of thousands of dollars, and the two analysts will now spend an
afternoon arguing about whose is right. Before taking sides, look closer.

In [ ]:
%sql
-- How many orders have more than one refund event? Each one is duplicated by the Finance join.
SELECT refund_events, COUNT(*) AS orders, ROUND(SUM(order_total), 2) AS order_total_duplicated
FROM (
  SELECT o.order_id, o.order_total, COUNT(r.refund_id) AS refund_events
  FROM lab_orders o JOIN lab_refunds r ON o.order_id = r.order_id
  GROUP BY o.order_id, o.order_total
)
GROUP BY refund_events
ORDER BY refund_events

In [ ]:
%sql
-- Two more things neither analyst filtered on
SELECT
  COUNT_IF(is_test_order)                                AS test_orders,
  ROUND(SUM(order_total) FILTER (WHERE is_test_order), 2) AS test_order_total,
  COUNT_IF(status = 'cancelled')                         AS cancelled_orders
FROM lab_orders
WHERE order_date >= current_date() - INTERVAL 90 DAYS

**What just happened**

- **Fan-out.** `lab_refunds` has one row per *refund event*. An order refunded in three instalments
  appears three times after the join, so Finance's `SUM(o.order_total)` counts its total three times.
  The refunds are subtracted correctly; the order totals are not.
- **Test orders.** Internal QA accounts place large orders every day. Neither query excludes them.
- **Gross vs. net.** Sales Ops never subtracts refunds at all.

So the argument is about a small gap between two numbers that are **both** wrong. The company
already wrote down what revenue means; nobody reads it:

In [ ]:
%sql
SELECT * FROM metric_glossary

## Step 3 — "How many active customers do we have?"

In [ ]:
%sql
-- Marketing team says: active = ordered in the last 30 days
SELECT COUNT(DISTINCT customer_id) AS active_customers
FROM lab_orders
WHERE order_date >= current_date() - INTERVAL 30 DAYS

In [ ]:
%sql
-- Customer Success team says: active = ordered in the last 90 days
SELECT COUNT(DISTINCT customer_id) AS active_customers
FROM lab_orders
WHERE order_date >= current_date() - INTERVAL 90 DAYS

Same story, bigger gap: about 30% of customers churn part-way through the data, so the
lookback window changes the answer by a thousand customers. Both teams are "right" by their own
definition. RevOps' definition (90 days, completed, non-test orders) is neither query.

## Step 4 — Fix the grain once

Before modelling, make refunds safe to join: one row per order. Every consumer that joins this
view instead of `lab_refunds` is immune to the fan-out.

In [ ]:
%sql
CREATE OR REPLACE VIEW order_refund_totals
COMMENT 'Refunds rolled up to ONE ROW PER ORDER. Join this, never lab_refunds, to orders.'
AS SELECT
  order_id,
  SUM(refund_amount) AS refund_amount,
  COUNT(*)           AS refund_events
FROM lab_refunds
GROUP BY order_id

## Step 5 — Build the metric view

| YAML key | What it does here |
|---|---|
| `version: 1.1` | Spec version. 1.1 is needed for `synonyms` and `display_name`. |
| `source` | The fact: `lab_orders`. |
| `filter` | Applied to **every** query: `NOT is_test_order`. |
| `joins` | Many-to-one lookups: `customer`, `product` and the order-grain `refunds` rollup. Joined only when a query needs them. |
| `dimensions` | What you can group and filter by. Newer docs call these `fields`; both keywords work. |
| `measures` | Aggregations, each with its exclusions built in (`FILTER (WHERE status <> 'cancelled')`). |
| `comment`, `synonyms`, `display_name` | Agent metadata. Genie reads them; `revenue` is a synonym of `net_revenue`, **not** of `gross_revenue`. |

The helper function `semantic_model.create_metric_view` renders the YAML with your fully-qualified names and runs
`CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML AS $$ ... $$`. Read the YAML it prints.

In [ ]:
yaml_text = semantic_model.create_metric_view(show_yaml=True)

Good question — this step is doing something a bit unusual (SQL that contains YAML), so let me unpack it.

## Why a metric view's definition is YAML, not normal SQL

A regular Databricks view is just: `CREATE VIEW x AS SELECT ...`. A **metric view** is different — it needs to store a richer structure (measures, dimensions, joins, synonyms, comments) that doesn't fit into a single `SELECT`. Databricks' answer to that is: the view's body is a YAML document describing that structure, wrapped inside a normal `CREATE VIEW` statement:

```sql
CREATE OR REPLACE VIEW catalog.schema.sales_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: catalog.schema.lab_orders
filter: NOT is_test_order
joins:
  - name: customer
    source: catalog.schema.lab_customers
    'on': source.customer_id = customer.customer_id
  ...
measures:
  - name: net_revenue
    expr: SUM(order_total - COALESCE(refunds.refund_amount, 0)) FILTER (WHERE status <> 'cancelled')
    synonyms: ['revenue', 'sales', ...]
$$;
```

The `$$ ... $$` is just SQL's way of saying "everything between these markers is a literal string" — so the YAML text becomes the stored definition of the view. Unity Catalog parses that YAML to know what measures and dimensions exist.

After this cell runs, `sales_metrics` exists as a governed object in Unity Catalog, and Step 6 queries it with `MEASURE(net_revenue)` to show that no matter how you group it, you get one consistent number — which is the whole point of the exercise.

## Step 6 — Query it with `MEASURE()`

Measures must be wrapped in `MEASURE()`; dimensions are used like columns. `SELECT *` is not allowed.
Group by anything; the engine generates the joins and aggregation.

In [ ]:
%sql
-- Finance's breakdown, through the metric view: the refund join can no longer fan out
SELECT product_category, MEASURE(net_revenue) AS revenue_90d
FROM sales_metrics
WHERE order_date >= current_date() - INTERVAL 90 DAYS
GROUP BY product_category
ORDER BY revenue_90d DESC

In [ ]:
%sql
SELECT region, customer_segment,
       MEASURE(net_revenue)   AS net_revenue,
       MEASURE(gross_revenue) AS gross_bookings,
       MEASURE(order_count)   AS orders
FROM sales_metrics
WHERE order_date >= current_date() - INTERVAL 90 DAYS
GROUP BY ALL
ORDER BY net_revenue DESC

Step 6 is where you actually *use* the metric view you just built — querying it the way a normal analyst or BI tool would, and seeing the payoff of Step 5's work.


## What's new compared to ordinary SQL

**`MEASURE(...)` is required, not optional.** In a regular view you'd write `SUM(order_total)`. Here you write `MEASURE(net_revenue)` instead — you're not re-deriving the aggregation, you're asking the engine to compute the measure that's already defined inside the metric view (with its refund subtraction and cancelled-order exclusion baked in, from Step 5). You can't just do `SELECT *` or reference raw columns from the underlying tables; measures only evaluate correctly through `MEASURE()`.

**Dimensions behave like normal columns.** `product_category`, `region`, `customer_segment` — these are the `dimensions` you defined in the YAML. You `SELECT`, `WHERE`, and `GROUP BY` them exactly like columns in a regular view, even though under the hood they come from joined tables (`product_category` lives on `lab_products`, `region` on `lab_customers`).

**`GROUP BY ALL`** just means "group by every selected non-aggregated column" — shorthand for `GROUP BY region, customer_segment`. It's a Databricks SQL feature, not specific to metric views, but convenient here since you're grouping by two dimensions.

**Joins happen lazily, only for what you ask.** The first query never touches `customer`, so Unity Catalog doesn't join `lab_customers` at all — it only pulls in `product` for `product_category`. The second query needs `customer` but not `product`. You don't write any `JOIN` clauses; the metric view's `joins` definition handles that invisibly based on which dimensions/measures you reference.

## The actual point of this step

This is the direct payoff of fixing the semantic layer. Recall from Steps 2–4: Sales Ops and Finance's raw queries disagreed, partly because of the refund fan-out (an order with 3 refund events got counted 3 times when joined naively).

Now look at the second query: it computes `net_revenue` (refund-adjusted) and `gross_revenue` (not refund-adjusted) *side by side*, by region and segment, and the refund join can no longer fan out — because the join is to the order-grain refund rollup you built in Step 4, not to the raw `lab_refunds` table.

The deeper point: however you slice this (by category, by region+segment, total, whatever), `net_revenue` always comes from the same definition. There's no way for one query to "accidentally" compute revenue a different way than another — which is exactly what Step 7 (reconciliation) verifies numerically: the total and the sum of the breakdown always match.

## Step 7 — Reconcile

`reconcile()` runs all four raw queries and four metric-view queries and saves every number to
`metric_reconciliation`. Each question is asked twice through the metric view, once as a total and
once as a breakdown that is summed back up. Through a semantic layer, both must give the same number.

In [ ]:
semantic_model.reconcile()
display(spark.table("metric_reconciliation").select("question", "approach", "value"))

Step 7 is the "prove it" step — it's where the notebook stops *telling* you the semantic layer fixed the inconsistency and actually *shows* you the numbers lining up.

## What the Reconcile does

`reconcile()` (in `semantic_model.py`) runs **eight queries** — the same two business questions from Step 2–3, answered four different ways each — and saves every result into a table called `metric_reconciliation`:

| question | approach | what it runs |
|---|---|---|
| `revenue_90d` | `raw_sales_ops` | the Sales Ops raw query from Step 2 |
| `revenue_90d` | `raw_finance` | the Finance raw query from Step 2 (the fan-out one) |
| `revenue_90d` | `metric_view_total` | `MEASURE(net_revenue)` with no grouping |
| `revenue_90d` | `metric_view_sum_of_categories` | `MEASURE(net_revenue)` grouped by category, then summed back up |
| `active_customers` | `raw_marketing_30d` | Marketing's 30-day definition from Step 3 |
| `active_customers` | `raw_cs_90d` | Customer Success's 90-day definition from Step 3 |
| `active_customers` | `metric_view_total` | `MEASURE(active_customers)` with no grouping |
| `active_customers` | `metric_view_sum_of_regions` | `MEASURE(active_customers)` grouped by region, then summed back up |

It writes all eight rows to `metric_reconciliation`, then the `display(...)` call shows you that table so you can read the comparison directly in the notebook output.

## Why it checks "total vs. sum of breakdown"

This is the key test of a semantic layer, not just "does it give one number" but "does that number stay the same no matter how you slice it." A metric view is only trustworthy if:

```
MEASURE(net_revenue) with no grouping
    ==
sum of MEASURE(net_revenue) grouped by category
```

If those two disagreed, it would mean the metric view's definition is inconsistent depending on the query shape — which would defeat the whole point. So `metric_view_sum_of_categories` and `metric_view_sum_of_regions` exist specifically to catch that failure mode. (This is also exactly what `test_metric_view_revenue_is_one_correct_number` checks in the grading, down to that same total-vs-breakdown comparison.)

## What you're meant to see in the output

Two raw numbers that disagree with each other (Sales Ops vs. Finance; Marketing vs. CS), sitting right next to two metric-view numbers that are identical regardless of whether they're totals or breakdowns-summed-up. That's the "before and after" picture: before the semantic layer, the same business question has multiple defensible-looking answers; after it, there's exactly one number, and slicing it differently doesn't change it.

That's also why this table exists as a saved object and not just printed text — it's evidence the automated checks can inspect later without recomputing anything themselves, and it's something you can point back to in Step 8 when you look at what Genie will eventually read from this same metric view.

## Step 8 — What Genie will read

Everything you wrote into the YAML (comments, synonyms, display names) is stored in Unity Catalog
and returned by `DESCRIBE`. This is the context a Genie Agent and Genie Ontology work from in
Exercises 2–4.

In [ ]:
%sql
DESCRIBE TABLE EXTENDED sales_metrics AS JSON

---

## Challenge 1

Extend the metric view with:

| Name | Kind | Definition |
|---|---|---|
| `order_month` | dimension | `order_date` truncated to the calendar month |
| `refund_rate` | measure | refunds on completed orders ÷ gross bookings of completed orders, as a ratio between 0 and 1 |

Write the YAML entries in the cell below (same shape as the entries printed in Step 5; indentation is
normalised for you), run it, then run the verification query.

`create_metric_view` replaces the whole definition. If you re-run Step 5 later, re-run this cell too.

**TO DO**

In [ ]:
extra_dimensions = """
- name: order_month
  expr: DATE_TRUNC('MONTH', order_date)
  comment: "Calendar month of the order date."
  display_name: Order Month
  synonyms: ['month']
"""

extra_measures = """
- name: refund_rate
  expr: SUM(COALESCE(refunds.refund_amount, 0)) FILTER (WHERE status <> 'cancelled') / SUM(order_total) FILTER (WHERE status <> 'cancelled')
  comment: "Share of gross bookings refunded. Ratio between 0 and 1."
  display_name: Refund Rate
  synonyms: ['return rate', 'refund percentage']
"""

semantic_model.create_metric_view(extra_dimensions=extra_dimensions, extra_measures=extra_measures)

In [ ]:
%sql
SELECT order_month,
       MEASURE(net_revenue) AS net_revenue,
       MEASURE(refund_rate) AS refund_rate
FROM sales_metrics
GROUP BY ALL
ORDER BY order_month

## Checks

In [ ]:
from helpers import test_runner
test_runner.run("01-semantic-layer-and-metric-views")